# CALE Current Experiments Visualization

This notebook summarizes the experiments we can support with the data already generated. It does **not** assume human/expert rubric annotations. The goal is to visualize the current internal construct probe: FEVER-derived adversarial factuality instances, target-model responses, evaluator variants, cross-framing behavior, boundary behavior, and CALE diagnostic subscores.

**Interpretation boundary.** These plots support preliminary internal construct-validity evidence. They should not be described as human-validated evaluator superiority unless a separate expert anchor set is added later.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 125,
    "savefig.dpi": 240,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
})

FRAMING_ORDER = ["neutral", "assertive", "authoritative", "polite_misleading"]
VARIANT_ORDER = [
    "baseline_binary",
    "baseline_likert",
    "direct_trustllm_heuristic",
    "generic_cale",
    "attack_aware_cale",
    "full_attack_aware_cale",
]
LABEL_ORDER = ["not_corrected", "uncertain", "partially_corrected", "corrected"]
CALE_VARIANTS = ["generic_cale", "attack_aware_cale", "full_attack_aware_cale"]

# Leave empty for auto-discovery on the server. If needed, paste exact report JSON paths here.
REPORT_PATHS = []

AUTO_REPORT_GLOBS = [
    "outputs/**/*_eval_report.json",
    "outputs/**/*_eval_summary_report.json",
    "outputs/**/*_stress_summary_report.json",
]

OUTDIR = Path("figures") / "current_experiments"
OUTDIR.mkdir(parents=True, exist_ok=True)

def infer_framing_from_path(path):
    name = Path(path).name
    for framing in sorted(FRAMING_ORDER, key=len, reverse=True):
        if f"_{framing}_" in name or framing in name:
            return framing
    return "unknown"

def infer_run_scale(n_predictions):
    if n_predictions >= 100000:
        return "full"
    if n_predictions >= 10000:
        return "medium"
    return "smoke"

def nice_name(value):
    return str(value).replace("Qwen/", "").replace("meta-llama/", "")

def nice_ylim(values, metric="score", min_span=0.04):
    values = pd.Series(values).dropna().astype(float)
    if values.empty:
        return (0, 1)
    ymin, ymax = values.min(), values.max()
    span = max(ymax - ymin, min_span)
    pad = span * 0.35
    lower, upper = ymin - pad, ymax + pad
    if metric not in {"delta", "score_shift"}:
        lower, upper = max(0, lower), min(1, upper)
    if upper <= lower:
        upper = lower + min_span
    return lower, upper

def annotate_points(ax, xs, ys, fmt="{:.3f}", dy=5):
    for x, y in zip(xs, ys):
        if pd.notna(y):
            ax.annotate(fmt.format(y), (x, y), textcoords="offset points", xytext=(0, dy), ha="center", fontsize=8)

def heatmap(df, index, columns, values, title, filename, cmap="viridis", vmin=None, vmax=None):
    if df.empty or values not in df.columns:
        print(f"Skipping {title}: no data for {values}.")
        return
    pivot = df.pivot_table(index=index, columns=columns, values=values, aggfunc="mean", observed=False)
    if index == "variant":
        pivot = pivot.reindex([v for v in VARIANT_ORDER if v in pivot.index])
    if columns == "framing":
        pivot = pivot[[c for c in FRAMING_ORDER if c in pivot.columns]]
    pivot = pivot.dropna(how="all")
    if pivot.empty:
        print(f"Skipping {title}: pivot is empty.")
        return
    fig, ax = plt.subplots(figsize=(max(7, 1.9 * len(pivot.columns)), max(3.8, 0.48 * len(pivot.index))))
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]
            label = "" if pd.isna(value) else f"{value:.3f}"
            ax.text(j, i, label, ha="center", va="center", color="white" if pd.notna(value) and value > np.nanmean(pivot.values) else "#111827", fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    fig.tight_layout()
    fig.savefig(OUTDIR / filename)
    plt.show()

## 1. Load Reports and Separate Experiment Scales

**What is computed.** The notebook auto-discovers `experiment.py` report JSON files and separates them by framing and prediction count.

**Why it matters.** Neutral full runs and cross-framing smoke runs should not be mixed in the same cross-framing plot. If both exist, full neutral is used for full internal-evaluation summaries, while the balanced 4-framing smoke group is used for framing-validity plots.

In [ ]:
if not REPORT_PATHS:
    paths = []
    for pattern in AUTO_REPORT_GLOBS:
        paths.extend(Path(".").glob(pattern))
    REPORT_PATHS = sorted({str(path) for path in paths})

if not REPORT_PATHS:
    raise FileNotFoundError("No report JSON files found. Run experiment.py first or set REPORT_PATHS manually.")

reports = []
pred_frames = []
metric_frames = []

for path in REPORT_PATHS:
    path = Path(path)
    report = json.loads(path.read_text(encoding="utf-8"))
    framing = infer_framing_from_path(path)
    rows = report.get("predictions", [])
    run_config = report.get("run_config", {})
    metrics = report.get("metrics", {})
    item = {
        "path": str(path),
        "framing": framing,
        "n_predictions": len(rows),
        "run_scale": infer_run_scale(len(rows)),
        **run_config,
    }
    reports.append(item)
    if rows:
        df = pd.DataFrame(rows)
        df["report_path"] = str(path)
        df["framing"] = framing
        if "model_name" in df.columns:
            df["model_short"] = df["model_name"].map(nice_name)
        pred_frames.append(df)
    for variant, values in metrics.items():
        row = {"report_path": str(path), "framing": framing, "variant": variant}
        row.update(values)
        metric_frames.append(row)

run_df_all = pd.DataFrame(reports)
pred_df_all = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
metrics_df_all = pd.DataFrame(metric_frames)

display_cols = [c for c in ["framing", "run_scale", "path", "n_predictions", "dataset_path", "target_models", "variants"] if c in run_df_all.columns]
display(run_df_all[display_cols].sort_values(["n_predictions", "framing"], ascending=[False, True]))

fig, ax = plt.subplots(figsize=(10, 3.8))
plot_df = run_df_all.sort_values("n_predictions", ascending=False).copy()
labels = [f"{r.framing}\n{Path(r.path).name[:28]}..." for r in plot_df.itertuples()]
ax.bar(range(len(plot_df)), plot_df["n_predictions"], color="#2563EB")
ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("prediction rows")
ax.set_title("Loaded Report Inventory")
for i, value in enumerate(plot_df["n_predictions"]):
    ax.text(i, value, f"{int(value):,}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(OUTDIR / "report_inventory.png")
plt.show()

## 2. Define the Current Experiment Sets

**Full internal evaluation.** Prefer the largest neutral report, usually the full FEVER dev internal evaluation.

**Cross-framing validation.** Prefer the prediction-count group that covers the most framing conditions, usually the 500-sample smoke group with four framings.

In [ ]:
full_candidates = run_df_all[run_df_all["framing"] == "neutral"].sort_values("n_predictions", ascending=False)
full_report_paths = set(full_candidates.head(1)["path"]) if not full_candidates.empty else set()

framing_candidates = (
    run_df_all.groupby("n_predictions")
    .agg(
        n_reports=("path", "count"),
        n_framings=("framing", "nunique"),
        framings=("framing", lambda values: ", ".join(sorted(set(map(str, values))))),
    )
    .reset_index()
    .sort_values(["n_framings", "n_predictions"], ascending=[False, False])
)
display(framing_candidates)

framing_report_paths = set()
if not framing_candidates.empty:
    selected_n = int(framing_candidates.iloc[0]["n_predictions"])
    framing_report_paths = set(run_df_all.loc[run_df_all["n_predictions"] == selected_n, "path"])
    print(f"Selected cross-framing group: n_predictions={selected_n:,}; reports={len(framing_report_paths)}")

full_pred_df = pred_df_all[pred_df_all["report_path"].isin(full_report_paths)].copy() if not pred_df_all.empty else pd.DataFrame()
full_metrics_df = metrics_df_all[metrics_df_all["report_path"].isin(full_report_paths)].copy() if not metrics_df_all.empty else pd.DataFrame()
framing_pred_df = pred_df_all[pred_df_all["report_path"].isin(framing_report_paths)].copy() if not pred_df_all.empty else pd.DataFrame()
framing_metrics_df = metrics_df_all[metrics_df_all["report_path"].isin(framing_report_paths)].copy() if not metrics_df_all.empty else pd.DataFrame()

print("Full internal predictions:", f"{len(full_pred_df):,}")
print("Cross-framing predictions:", f"{len(framing_pred_df):,}")
print("Full report path:")
for path in sorted(full_report_paths):
    print(" ", path)
print("Framing report paths:")
for path in sorted(framing_report_paths):
    print(" ", path)

## 3. Full Internal Evaluation: Variant Comparison

**What is computed.** For the largest neutral report, compare evaluator variants on shared outcome metrics: final mean score, label distribution, and NEI overclaim.

**Why it matters.** This is the cleanest current evidence for how direct baselines and CALE variants behave on the same generated response set. It does not prove human agreement, but it shows how evaluator design changes the measurement output.

In [ ]:
if full_pred_df.empty:
    print("No full neutral report found. Skipping full internal evaluation plots.")
else:
    full_summary = (
        full_pred_df.groupby("variant", dropna=False)
        .agg(n=("id", "count"), mean_score=("score", "mean"), mean_uncertainty=("uncertainty", "mean"))
        .reset_index()
    )
    if "reference_label" in full_pred_df.columns:
        nei = full_pred_df[full_pred_df["reference_label"] == "NOT ENOUGH INFO"].copy()
        if not nei.empty:
            nei["overclaim"] = nei["label"].isin(["corrected", "partially_corrected"]).astype(float)
            full_summary = full_summary.merge(nei.groupby("variant")["overclaim"].mean().rename("nei_overclaim_rate").reset_index(), on="variant", how="left")
    full_summary["variant"] = pd.Categorical(full_summary["variant"], categories=VARIANT_ORDER, ordered=True)
    full_summary = full_summary.sort_values("variant")
    display(full_summary)
    full_summary.to_csv(OUTDIR / "full_internal_variant_summary.csv", index=False)

    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    colors = ["#94A3B8" if str(v).startswith("baseline") or str(v).startswith("direct") else "#0F766E" for v in full_summary["variant"]]
    ax.bar(full_summary["variant"].astype(str), full_summary["mean_score"], color=colors)
    ax.set_title("Full Internal Evaluation: Mean Score by Evaluator Variant")
    ax.set_ylabel("mean score")
    ax.set_ylim(*nice_ylim(full_summary["mean_score"], "mean_score"))
    ax.tick_params(axis="x", rotation=25)
    for i, value in enumerate(full_summary["mean_score"]):
        ax.text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTDIR / "full_internal_mean_score_by_variant.png")
    plt.show()

    if "nei_overclaim_rate" in full_summary.columns:
        fig, ax = plt.subplots(figsize=(9.5, 4.2))
        ax.bar(full_summary["variant"].astype(str), full_summary["nei_overclaim_rate"], color="#B45309")
        ax.set_title("Full Internal Evaluation: NEI Overclaim Rate (lower is better)")
        ax.set_ylabel("NEI overclaim rate")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=25)
        for i, value in enumerate(full_summary["nei_overclaim_rate"]):
            if pd.notna(value):
                ax.text(i, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
        fig.tight_layout()
        fig.savefig(OUTDIR / "full_internal_nei_overclaim_by_variant.png")
        plt.show()

## 4. Cross-Framing Shared Metrics

**What is computed.** For each `framing × evaluator variant`, compute mean score, mean uncertainty, and NEI overclaim.

**Why it matters.** These are shared metrics. They can be compared across baseline and CALE variants because every variant outputs final score and final label.

In [ ]:
if framing_pred_df.empty:
    print("No balanced cross-framing reports found. Skipping cross-framing shared metrics.")
else:
    framing_summary = (
        framing_pred_df.groupby(["framing", "variant"], dropna=False)
        .agg(n=("id", "count"), mean_score=("score", "mean"), mean_uncertainty=("uncertainty", "mean"))
        .reset_index()
    )
    if "reference_label" in framing_pred_df.columns:
        nei = framing_pred_df[framing_pred_df["reference_label"] == "NOT ENOUGH INFO"].copy()
        if not nei.empty:
            nei["overclaim"] = nei["label"].isin(["corrected", "partially_corrected"]).astype(float)
            framing_summary = framing_summary.merge(nei.groupby(["framing", "variant"])["overclaim"].mean().rename("nei_overclaim_rate").reset_index(), on=["framing", "variant"], how="left")
    framing_summary["framing"] = pd.Categorical(framing_summary["framing"], categories=FRAMING_ORDER, ordered=True)
    framing_summary["variant"] = pd.Categorical(framing_summary["variant"], categories=VARIANT_ORDER, ordered=True)
    framing_summary = framing_summary.sort_values(["variant", "framing"])
    display(framing_summary)
    framing_summary.to_csv(OUTDIR / "cross_framing_shared_metrics.csv", index=False)

    def plot_metric_by_framing(metric, title, ylabel, filename, lower_is_better=False):
        if metric not in framing_summary.columns or framing_summary[metric].dropna().empty:
            print(f"Skipping {metric}: no values.")
            return
        fig, ax = plt.subplots(figsize=(11, 5))
        for variant, sub in framing_summary.dropna(subset=[metric]).groupby("variant", observed=True):
            sub = sub.sort_values("framing")
            xs = sub["framing"].astype(str).tolist()
            ys = sub[metric].tolist()
            ax.plot(xs, ys, marker="o", linewidth=2, label=str(variant))
            annotate_points(ax, xs, ys)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel("framing")
        ax.set_ylim(0, 1 if metric == "nei_overclaim_rate" else nice_ylim(framing_summary[metric], metric)[1])
        if metric != "nei_overclaim_rate":
            ax.set_ylim(*nice_ylim(framing_summary[metric], metric))
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
        fig.tight_layout()
        fig.savefig(OUTDIR / filename)
        plt.show()

    plot_metric_by_framing("mean_score", "Mean Evaluator Score Across Framing", "mean score", "cross_framing_mean_score.png")
    plot_metric_by_framing("nei_overclaim_rate", "NEI Overclaim Rate Across Framing (lower is better)", "NEI overclaim rate", "cross_framing_nei_overclaim.png", lower_is_better=True)

## 5. Cross-Framing Stability Relative to Neutral

**What is computed.** Match rows by `id × model_name × variant` and compute score shift and label flip relative to neutral.

**Why it matters.** This is the core discriminant-validity check: when the factual core stays fixed, evaluator judgments should not swing only because rhetorical framing changes.

In [ ]:
stability_df = pd.DataFrame()
if framing_pred_df.empty:
    print("No cross-framing predictions found.")
else:
    key_cols = ["id", "model_name", "variant"]
    if not set(key_cols).issubset(framing_pred_df.columns):
        print("Missing required columns for matched stability:", set(key_cols) - set(framing_pred_df.columns))
    else:
        neutral = framing_pred_df[framing_pred_df["framing"] == "neutral"][[*key_cols, "score", "label"]].rename(columns={"score": "neutral_score", "label": "neutral_label"})
        non_neutral = framing_pred_df[framing_pred_df["framing"] != "neutral"][[*key_cols, "framing", "score", "label", "reference_label"]]
        matched = non_neutral.merge(neutral, on=key_cols, how="inner")
        if matched.empty:
            print("No matched neutral/non-neutral rows found.")
        else:
            matched["score_shift"] = matched["score"] - matched["neutral_score"]
            matched["abs_score_shift"] = matched["score_shift"].abs()
            matched["label_flip"] = matched["label"] != matched["neutral_label"]
            stability_df = (
                matched.groupby(["framing", "variant"], dropna=False)
                .agg(n=("id", "count"), mean_score_shift=("score_shift", "mean"), mean_abs_score_shift=("abs_score_shift", "mean"), label_flip_rate=("label_flip", "mean"))
                .reset_index()
            )
            stability_df["framing"] = pd.Categorical(stability_df["framing"], categories=FRAMING_ORDER, ordered=True)
            stability_df["variant"] = pd.Categorical(stability_df["variant"], categories=VARIANT_ORDER, ordered=True)
            stability_df = stability_df.sort_values(["variant", "framing"])
            display(stability_df)
            stability_df.to_csv(OUTDIR / "cross_framing_stability.csv", index=False)
            heatmap(stability_df, "variant", "framing", "mean_score_shift", "Mean Score Shift from Neutral", "heatmap_mean_score_shift.png", cmap="coolwarm", vmin=-0.05, vmax=0.05)
            heatmap(stability_df, "variant", "framing", "label_flip_rate", "Label Flip Rate from Neutral", "heatmap_label_flip_rate.png", cmap="magma", vmin=0, vmax=1)

## 6. Boundary Behavior by FEVER Label

**What is computed.** Use FEVER labels as boundary conditions, not as response-level expert labels. For NEI, compute overclaim. For REFUTES, compute whether the evaluator gives correction credit. For SUPPORTS, compute over-correction / uncertainty.

**Why it matters.** These checks directly test whether the evaluator behaves sensibly under falsehood, true-control, and insufficient-evidence cases.

In [ ]:
boundary_df = pd.DataFrame()
source_df = framing_pred_df if not framing_pred_df.empty else full_pred_df
if source_df.empty or "reference_label" not in source_df.columns:
    print("No reference_label available for boundary analysis.")
else:
    boundary = source_df.copy()
    boundary["nei_overclaim_rate"] = np.where(
        boundary["reference_label"] == "NOT ENOUGH INFO",
        boundary["label"].isin(["corrected", "partially_corrected"]).astype(float),
        np.nan,
    )
    boundary["refutes_correction_credit"] = np.where(
        boundary["reference_label"] == "REFUTES",
        boundary["label"].isin(["corrected", "partially_corrected"]).astype(float),
        np.nan,
    )
    boundary["supports_overcorrection_rate"] = np.where(
        boundary["reference_label"] == "SUPPORTS",
        boundary["label"].isin(["not_corrected", "uncertain"]).astype(float),
        np.nan,
    )
    group_cols = ["variant"] + (["framing"] if "framing" in boundary.columns and boundary["framing"].nunique() > 1 else [])
    boundary_df = (
        boundary.groupby(group_cols, dropna=False)
        .agg(
            nei_overclaim_rate=("nei_overclaim_rate", "mean"),
            refutes_correction_credit=("refutes_correction_credit", "mean"),
            supports_overcorrection_rate=("supports_overcorrection_rate", "mean"),
        )
        .reset_index()
    )
    if "framing" in boundary_df.columns:
        boundary_df["framing"] = pd.Categorical(boundary_df["framing"], categories=FRAMING_ORDER, ordered=True)
    boundary_df["variant"] = pd.Categorical(boundary_df["variant"], categories=VARIANT_ORDER, ordered=True)
    boundary_df = boundary_df.sort_values([c for c in ["variant", "framing"] if c in boundary_df.columns])
    display(boundary_df)
    boundary_df.to_csv(OUTDIR / "boundary_behavior.csv", index=False)
    if "framing" in boundary_df.columns:
        heatmap(boundary_df, "variant", "framing", "nei_overclaim_rate", "NEI Overclaim by Variant and Framing", "heatmap_nei_overclaim.png", cmap="inferno", vmin=0, vmax=1)
        heatmap(boundary_df, "variant", "framing", "refutes_correction_credit", "REFUTES Correction Credit", "heatmap_refutes_credit.png", cmap="viridis", vmin=0, vmax=1)
        heatmap(boundary_df, "variant", "framing", "supports_overcorrection_rate", "SUPPORTS Over-Correction / Uncertainty", "heatmap_supports_overcorrection.png", cmap="magma", vmin=0, vmax=1)

## 7. CALE Diagnostic Subscores

**What is computed.** Average CALE rubric subscores by dimension and framing, usually focusing on `full_attack_aware_cale`.

**Why it matters.** These are not baseline-vs-CALE shared outcome metrics. Their role is interpretability: they show which construct dimensions are bottlenecks.

In [ ]:
subscore_rows = []
source_df = framing_pred_df if not framing_pred_df.empty else full_pred_df
for row in source_df.to_dict("records"):
    subscores = row.get("subscores") or {}
    if not isinstance(subscores, dict):
        continue
    for dimension, value in subscores.items():
        subscore_rows.append({
            "id": row.get("id"),
            "model_short": row.get("model_short", row.get("model_name")),
            "variant": row.get("variant"),
            "framing": row.get("framing", "neutral"),
            "reference_label": row.get("reference_label"),
            "dimension": dimension,
            "score": float(value),
        })
subscore_df = pd.DataFrame(subscore_rows)
if subscore_df.empty:
    print("No CALE subscores found.")
else:
    chosen_variant = "full_attack_aware_cale" if "full_attack_aware_cale" in set(subscore_df["variant"]) else subscore_df["variant"].iloc[0]
    construct = subscore_df[subscore_df["variant"] == chosen_variant].copy()
    construct_summary = construct.groupby(["dimension", "framing"], dropna=False)["score"].mean().reset_index()
    display(construct_summary.sort_values(["framing", "score"]))
    construct_summary.to_csv(OUTDIR / "cale_construct_subscores.csv", index=False)
    heatmap(construct_summary, "dimension", "framing", "score", f"CALE Construct Subscores Across Framing ({chosen_variant})", "heatmap_cale_construct_subscores.png", cmap="YlGnBu", vmin=0, vmax=1)
    bottleneck = construct_summary.sort_values(["framing", "score"]).groupby("framing").head(4)
    display(bottleneck)
    bottleneck.to_csv(OUTDIR / "cale_construct_bottlenecks.csv", index=False)

## 8. PPT-Ready Takeaway Table

**Use this table in the report/PPT.** It keeps claims conservative: current results are internal construct-probe evidence, not human-validated superiority.

In [ ]:
takeaways = []
if "full_summary" in globals() and not full_pred_df.empty:
    top_direct = full_summary[full_summary["variant"].astype(str) == "direct_trustllm_heuristic"]
    full_cale = full_summary[full_summary["variant"].astype(str) == "full_attack_aware_cale"]
    if not top_direct.empty and not full_cale.empty:
        takeaways.append(["Full neutral internal evaluation", "Direct holistic baseline gives higher mean scores; Full CALE is more conservative and decomposes the judgment into dimensions."])
if "framing_summary" in globals() and not framing_pred_df.empty:
    takeaways.append(["Cross-framing smoke validation", "Framing changes induce measurable but not catastrophic score shifts; CALE variants tend to become more conservative under assertive framing."])
if not boundary_df.empty:
    takeaways.append(["Boundary behavior", "NEI overclaim remains high, making unsupported correction the main current failure mode and a strong motivation for calibration."])
if "construct_summary" in globals() and not subscore_df.empty:
    lowest = construct_summary.sort_values("score").head(3)
    low_dims = ", ".join(lowest["dimension"].astype(str).tolist())
    takeaways.append(["CALE diagnostics", f"Lowest construct dimensions are concentrated around: {low_dims}. This supports the value of dimension-level diagnostics."])
takeaway_df = pd.DataFrame(takeaways, columns=["Evidence block", "Conservative interpretation"])
display(takeaway_df)
takeaway_df.to_csv(OUTDIR / "ppt_ready_takeaways.csv", index=False)
print("Saved PPT/report-ready figures and tables to:", OUTDIR)